# 04 Statistical Analysis\nPurpose: Quantify significance and direction of risk factors using formal statistical methods.

In [ ]:
from pathlib import Path\nimport pandas as pd\nimport numpy as np\nfrom scipy.stats import chi2_contingency, ttest_ind\nimport statsmodels.api as sm\n\nROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()\nCLEAN_PATH = ROOT / 'data' / 'processed' / 'loan_clean.csv'\nCHI_OUT = ROOT / 'reports' / 'phase_04_chi_square_results.csv'\nCOEF_OUT = ROOT / 'reports' / 'phase_04_logit_coefficients.csv'

In [ ]:
df = pd.read_csv(CLEAN_PATH)\ndf.shape

In [ ]:
# Chi-square test for categorical features\ncat_cols = ['Education','EmploymentType','MaritalStatus','HasMortgage','HasDependents','LoanPurpose','HasCoSigner']\nchi_rows = []\nfor col in cat_cols:\n    table = pd.crosstab(df[col], df['Default'])\n    chi2, pval, dof, _ = chi2_contingency(table)\n    chi_rows.append({'feature': col, 'chi2': chi2, 'p_value': pval, 'dof': dof})\nchi_df = pd.DataFrame(chi_rows).sort_values('p_value')\nchi_df.to_csv(CHI_OUT, index=False)\nchi_df

In [ ]:
# Two-sample t-test for numeric comparison\nd1 = df.loc[df['Default'] == 1, 'InterestRate']\nd0 = df.loc[df['Default'] == 0, 'InterestRate']\nt_stat, p_val = ttest_ind(d1, d0, equal_var=False)\nprint({'metric': 'InterestRate_by_Default', 't_stat': t_stat, 'p_value': p_val})

In [ ]:
num_cols = ['Age','Income','LoanAmount','CreditScore','MonthsEmployed','NumCreditLines','InterestRate','LoanTerm','DTIRatio','IncomeToLoanRatio']\nX = pd.get_dummies(df[num_cols + cat_cols], drop_first=True)\nX = sm.add_constant(X)\ny = df['Default']\nmodel = sm.Logit(y, X).fit(disp=False)\ncoef_df = pd.DataFrame({\n    'feature': model.params.index,\n    'coefficient': model.params.values,\n    'p_value': model.pvalues.values,\n})\ncoef_df.to_csv(COEF_OUT, index=False)\ncoef_df.sort_values('p_value').head(20)